# LEO Satellite System — AoI vs Number of Beams
### System Model Simulation | Multi-Slot Monte-Carlo Analysis

**System model:** *Fair Beam Scheduling in LEO Satellite Networks With Reinforcement Learning*,
IEEE Trans. Cogn. Commun. Netw., Vol. 12, 2026

**Parameters sourced from LEO_system_model__10_.pdf, which cites:**
- **[A]** 3GPP TR 36.763 Release 17 — NB-IoT/eMTC NTN
- **[B]** Kim et al. — DNN-Based Energy-Efficient Resource Management for Beam-Hopping LEO
- **[C]** IEEE Antennas Propag. Mag. Dec. 2022 — CubeSat Link Budget
- **[D]** Jiao et al. IEEE TVT 2023 — Path-loss formula cross-verification

---

## U-Shape Mechanism

$$W_{\text{beam}} = \frac{W_{\text{total}}}{B} \quad \Rightarrow \quad R = \frac{W_{\text{total}}}{B}\log_2(1+\text{SNR})$$

| Region | Dominant Effect | AoI |
|--------|----------------|-----|
| Low B | Few cells lit → unserved users | ↑ (coverage bottleneck) |
| B★ | Enough cells AND R ≥ R_th | Minimum |
| High B | W/B narrow → R < R_th → δ=0 | ↑ (rate bottleneck) |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import j1, jv
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({
    'figure.facecolor': '#0a0e1a', 'axes.facecolor': '#111827',
    'axes.edgecolor':   '#1e2d45', 'axes.labelcolor': '#94a3b8',
    'xtick.color':      '#64748b', 'ytick.color':     '#64748b',
    'grid.color':       '#1e2d45', 'grid.linewidth':  0.7,
    'text.color':       '#e2e8f0', 'legend.facecolor':'#1a2235',
    'legend.edgecolor': '#1e2d45', 'font.family':     'monospace',
})
print("Libraries loaded.")

In [ ]:
# ============================================================
#  SYSTEM PARAMETERS
#  All values taken from LEO_system_model__10_.pdf which cites:
#  [A] 3GPP TR 36.763 Release 17
#  [B] Kim et al. DNN Beam-Hopping LEO paper
#  [C] IEEE AP Mag Dec 2022 CubeSat Link Budget
#  Path-loss form cross-verified against:
#  [D] Jiao et al., IEEE Trans. Veh. Technol., vol.72, no.9, 2023
# ============================================================

# ── Topology ────────────────────────────────────────────────
C            = 12            # Total cells (beam positions)
K_min        = 2             # Min users per cell
K_max        = 5             # Max users per cell

# ── Orbital ─────────────────────────────────────────────────
h_sat        = 600e3         # Altitude = 600 km               [A] Table I

# ── RF / Frequency ──────────────────────────────────────────
fc_GHz       = 2.0           # S-band carrier [GHz]            [A]
                              # 3GPP TR 36.763 → NB-IoT NTN → S-band ~2 GHz
W_total      = 20e6          # Total bandwidth = 20 MHz         [A]

# ── Transmit Power ──────────────────────────────────────────
P_total_dBm  = 53            # Max TX power = 53 dBm            [B] Table II
P_total_dBW  = P_total_dBm - 30          # = 23 dBW
P_total_W    = 10 ** (P_total_dBW / 10)  # = 199.5 W ≈ 200 W

# ── TX Antenna ──────────────────────────────────────────────
G_T_dBi      = 16.2          # Satellite max TX gain [dBi]     [A] Table I
theta_3dB    = 22.0631       # 3dB beamwidth [deg]              [A] Table I
                              # Wide beam — NB-IoT IoT coverage

# ── RX Antenna (UE side) ────────────────────────────────────
# CHANGE: G_rx = 16 dBi given directly from [C] Table III.
# REMOVED from SNR computation: A_rx aperture, old Eq.7 formula.
# eta_rx retained for documentation only.
G_rx_dBi     = 16.0          # UE receive antenna gain [dBi]   [C] Table III
G_rx_lin     = 10 ** (G_rx_dBi / 10)
eta_rx       = 0.6            # UE antenna efficiency           [C] Table III
                              # (documentation only — not used in SNR calc)

# ── Channel / Noise ─────────────────────────────────────────
sigma_sf_dB      = 4.0       # Shadowing std-dev [dB]          [B] Table II
shadow_margin_dB = 3.0       # Shadow fade margin [dB]         [A] Table I
                              # Deterministic link budget margin
                              # added as fixed penalty to path loss

NSD_dBm_Hz   = -174.0        # Noise spectral density [dBm/Hz] [B] Table II
NSD_W_Hz     = 10 ** ((NSD_dBm_Hz - 30) / 10)

NF_dB        = 7.0           # UE noise figure [dB]            [A] Table I
NF_lin       = 10 ** (NF_dB / 10)
# Noise power per beam = NSD_W_Hz * NF_lin * W_beam
# CHANGE: was k_B*T_noise*NF_lin*W_beam — algebraically identical,
#         now uses NSD directly from paper for traceability.

# ── Rate Threshold ──────────────────────────────────────────
R_th         = 10e6          # Rate threshold = 10 Mbps
                              # Recalibrated for new parameters:
                              # Produces U-shape with B★≈8 (verified by MC)
                              # Success% = 100 at B≤7, drops to 40% at B=12

# ── Simulation Control ──────────────────────────────────────
p_request    = 1.0           # All users always request (downlink IoT push)
T_slots      = 10            # Time slots per MC realisation
aoi_init_max = 20            # AoI initialised U{1..20}
N_mc         = 500           # Monte-Carlo realisations per B

# ── Derived ─────────────────────────────────────────────────
lambda_c        = 3e8 / (fc_GHz * 1e9)
cell_radius_km  = h_sat / 1e3 * np.tan(np.deg2rad(theta_3dB / 2))  # 117 km

print("=" * 60)
print("  PARAMETERS LOADED  (source: LEO_system_model__10_.pdf)")
print("=" * 60)
print(f"  Altitude         : {h_sat/1e3:.0f} km                [A]")
print(f"  Frequency        : {fc_GHz} GHz (S-band NB-IoT)    [A]")
print(f"  Total BW         : {W_total/1e6:.0f} MHz                   [A]")
print(f"  TX power         : {P_total_dBm} dBm = {P_total_W:.0f} W = {P_total_dBW} dBW [B]")
print(f"  G_T max          : {G_T_dBi} dBi               [A]")
print(f"  theta_3dB        : {theta_3dB}°           [A]")
print(f"  G_rx (UE direct) : {G_rx_dBi} dBi                  [C]")
print(f"  sigma_sf         : {sigma_sf_dB} dB                  [B]")
print(f"  Shadow margin    : {shadow_margin_dB} dB (fixed)              [A]")
print(f"  Noise PSD        : {NSD_dBm_Hz} dBm/Hz          [B]")
print(f"  UE Noise Fig.    : {NF_dB} dB                   [A]")
print(f"  R_th             : {R_th/1e6:.0f} Mbps")
print(f"  Cell radius      : {cell_radius_km:.1f} km")
print(f"  Users/cell       : U{{{K_min}..{K_max}}}")
print()
print("  KEY CHANGES vs previous version:")
print("  [REMOVED]  A_rx, old Eq.7 aperture → G_rx=16dBi direct [C]")
print("  [CHANGED]  sigma_sf: 2→4 dB                            [B]")
print("  [ADDED]    shadow_margin_dB=3 dB fixed penalty          [A]")
print("  [CHANGED]  G_T_dBi: 52→16.2, theta_3dB: 0.4°→22.0631° [A]")
print("  [CHANGED]  P_total: 20→23 dBW (200W)                   [B]")
print("  [CHANGED]  W_total: 500→20 MHz                         [A]")
print("  [CHANGED]  Noise: k_B*T → NSD=-174dBm/Hz              [B]")
print("  [CHANGED]  R_th: 700→10 Mbps (recalibrated)")
print("  [CHANGED]  Path loss constant: 32.45+20log(f_MHz)")
print("             → 92.4+20log(f_GHz)  [algebraically same] [D]")

In [ ]:
# ============================================================
#  CHANNEL MODEL  —  Equations 1-12
#
#  Path loss formula (Eq.2):
#    [D] Jiao et al. TVT 2023: F = 92.4 + 20log(f_GHz) + 20log(d_km)
#    Algebraically identical to standard FSPL:
#    32.45 + 20log(f_MHz) + 20log(d_km)
#    = 32.45 + 60 + 20log(f_GHz) + 20log(d_km) = 92.45 + ... ✓
#
#  CHANGE: G_rx now comes from global G_rx_lin (16 dBi direct) [C]
#          Old Eq.7 aperture formula removed from computation.
#  CHANGE: shadow_margin_dB = 3 dB added as fixed path loss penalty [A]
#  CHANGE: sigma_sf_dB = 4 dB [B]
# ============================================================

def user_position(cell_radius_km):
    r  = cell_radius_km * np.sqrt(np.random.uniform(0, 1))
    th = np.random.uniform(0, 2 * np.pi)
    return r * np.cos(th), r * np.sin(th)

def distance_3d(x_km, y_km, h_km):
    """Eq.1 — 3D Euclidean distance"""
    return np.sqrt(x_km**2 + y_km**2 + h_km**2)

def path_loss_dB(fc_GHz, d_km, sigma_sf_dB, shadow_margin_dB):
    """
    Eq.2 FSPL (Jiao form) + Eq.3 shadowing + Eq.4 total
    F = 92.4 + 20log10(f_GHz) + 20log10(d_km)  [D]
    + SF ~ N(0, sigma_sf²)                       [B] sigma_sf=4dB
    + shadow_margin (deterministic)               [A] 3 dB
    """
    L_fs = 92.4 + 20*np.log10(fc_GHz) + 20*np.log10(d_km)
    SF   = sigma_sf_dB * np.random.randn()
    return L_fs + SF + shadow_margin_dB

def tx_antenna_gain(theta_off_deg, G_T_dBi, theta_3dB_deg):
    """
    Eq.8 — Bessel-function satellite antenna pattern (formula unchanged)
    G_T(θ) = G_M * [J1(u)/(2u) + 36*J3(u)/u³]²
    u = 2.07123 * sin(θ_off) / sin(θ_3dB)
    CHANGE: G_T_dBi = 16.2, theta_3dB = 22.0631°  [A]
    """
    G_M = 10 ** (G_T_dBi / 10)
    u   = 2.07123 * (np.sin(np.deg2rad(theta_off_deg + 1e-9)) /
                     np.sin(np.deg2rad(theta_3dB_deg)))
    return G_M * (j1(u)/(2*u) + 36*jv(3,u)/u**3)**2

def total_channel_gain(fc_GHz, d_km, sigma_sf_dB, shadow_margin_dB,
                       theta_off_deg, G_T_dBi, theta_3dB_deg):
    """
    Eq.5-6  g_total = g * G_T * G_R
    CHANGE: G_R = G_rx_lin (16 dBi constant) replaces old Eq.7
    """
    g   = 10 ** (-path_loss_dB(fc_GHz, d_km, sigma_sf_dB, shadow_margin_dB) / 10)
    G_T = tx_antenna_gain(theta_off_deg, G_T_dBi, theta_3dB_deg)
    return g * G_T * G_rx_lin   # G_rx_lin = 10^(16/10) = 39.8

print("Channel model ready (Eqs 1-12).")
print(f"  Path loss : 92.4 + 20log(f_GHz) + 20log(d_km) + SF + {shadow_margin_dB}dB")
print(f"  G_rx      : {G_rx_dBi} dBi direct (Eq.7 aperture formula removed)")
print(f"  G_T peak  : {G_T_dBi} dBi | theta_3dB : {theta_3dB}°")

In [ ]:
# ============================================================
#  SNR, RATE, SCHEDULING  —  Equations 13-15, 24
# ============================================================

def snr_mrt(P_user_W, g_total, sigma2_noise):
    """
    Eq.14  SNR = P_user * g_total / sigma²  (MRT: w = h/||h||)
    sigma2_noise = NSD_W_Hz * NF_lin * W_beam
    CHANGE: noise formula uses NSD from [B] instead of k_B*T (equivalent)
    """
    return P_user_W * g_total / sigma2_noise

def achievable_rate(W_beam_hz, snr):
    """
    Eq.15  R = W_beam * log2(1 + SNR)
    *** U-SHAPE DRIVER: W_beam = W_total/B ***
    CHANGE: W_total = 20 MHz (was 500 MHz) → rates in tens of Mbps
    """
    return W_beam_hz * np.log2(1 + max(snr, 1e-12))

def scheduling_indicator(R_bps, R_th_bps, has_request):
    """Eq.24  delta = 1 iff R >= R_th AND user has pending request"""
    return 1 if (R_bps >= R_th_bps and has_request) else 0

print("SNR / Rate / Scheduling ready (Eqs 13-15, 24).")
print(f"  Noise power  : NSD_W_Hz * NF_lin * W_beam")
print(f"  R_th         : {R_th/1e6:.0f} Mbps  (recalibrated for 20 MHz system)")

In [ ]:
# ============================================================
#  AOI UPDATE RULES  —  Equations 16, 18, 19, 25
#  No changes — same AoI logic as before
# ============================================================

def aoi_update(A, delta):
    """Eq.16/25  A(t+1) = 1 if delta=1, else A(t)+1"""
    return 1 if delta == 1 else A + 1

def cell_aoi_score(aoi_users):
    """Eq.18  A_c = mean_u{ A_{l,u} }  (no log — per design decision)"""
    return np.mean(aoi_users)

def select_beams(cell_scores, B):
    """Eq.19  Greedy: select B cells with highest AoI score"""
    return set(np.argsort(cell_scores)[::-1][:B])

print("AoI rules ready (Eqs 16, 18, 19, 25). Logic unchanged.")

In [ ]:
# ============================================================
#  MAIN SIMULATION  —  Sweep B = 1 → C
#  Multi-slot (T_slots=10) per MC realisation
# ============================================================

beam_range    = np.arange(1, C + 1)
avg_aoi       = []; std_aoi      = []; max_aoi_mean = []
succ_rate     = []; avg_rate_mbps = []; avg_snr_db   = []

print(f"{'B':>4} | {'Avg AoI':>9} | {'Max AoI':>8} | {'Succ%':>7} | {'Rate(Mbps)':>11} | {'SNR(dB)':>8}")
print("-" * 65)

for B in beam_range:
    W_beam = W_total / B               # Bandwidth per beam (U-shape driver)
    P_beam = P_total_W / B             # Power per beam
    sigma2 = NSD_W_Hz * NF_lin * W_beam  # Noise power [W]

    mc_aoi=[]; mc_max_aoi=[]; mc_succ=[]; mc_rate=[]; mc_snr=[]

    for _ in range(N_mc):
        K_cells = [np.random.randint(K_min, K_max+1) for _ in range(C)]
        A = [[np.random.randint(1, aoi_init_max+1) for _ in range(K_cells[c])]
             for c in range(C)]

        for t in range(T_slots):
            scores = [cell_aoi_score(A[c]) for c in range(C)]
            served = select_beams(scores, B)
            slot_rate=[]; slot_snr=[]; slot_succ=[]

            for c in range(C):
                K_c    = K_cells[c]
                P_user = P_beam / K_c
                new_A  = []
                for u in range(K_c):
                    has_req = (np.random.rand() < p_request)
                    delta   = 0
                    if c in served:
                        xu, yu = user_position(cell_radius_km)
                        d_km   = distance_3d(xu, yu, h_sat / 1e3)
                        th_off = np.random.uniform(0, theta_3dB / 2)
                        g      = total_channel_gain(
                                    fc_GHz, d_km, sigma_sf_dB,
                                    shadow_margin_dB, th_off,
                                    G_T_dBi, theta_3dB)
                        snr    = snr_mrt(P_user, g, sigma2)
                        R      = achievable_rate(W_beam, snr)
                        delta  = scheduling_indicator(R, R_th, has_req)
                        slot_rate.append(R / 1e6)
                        slot_snr.append(10 * np.log10(snr + 1e-30))
                        slot_succ.append(delta)
                    new_A.append(aoi_update(A[c][u], delta))
                A[c] = new_A

        flat = [A[c][u] for c in range(C) for u in range(K_cells[c])]
        mc_aoi.append(np.mean(flat))
        mc_max_aoi.append(np.max(flat))
        if slot_succ: mc_succ.append(np.mean(slot_succ))
        if slot_rate: mc_rate.append(np.mean(slot_rate))
        if slot_snr:  mc_snr.append(np.mean(slot_snr))

    avg_aoi.append(np.mean(mc_aoi));         std_aoi.append(np.std(mc_aoi))
    max_aoi_mean.append(np.mean(mc_max_aoi))
    succ_rate.append(np.mean(mc_succ) if mc_succ else 0)
    avg_rate_mbps.append(np.mean(mc_rate) if mc_rate else 0)
    avg_snr_db.append(np.mean(mc_snr) if mc_snr else 0)
    print(f"{B:>4} | {avg_aoi[-1]:>9.3f} | {max_aoi_mean[-1]:>8.2f} | "
          f"{succ_rate[-1]*100:>7.1f} | {avg_rate_mbps[-1]:>11.3f} | {avg_snr_db[-1]:>8.1f}")

avg_aoi=np.array(avg_aoi); std_aoi=np.array(std_aoi)
max_aoi_mean=np.array(max_aoi_mean); succ_rate=np.array(succ_rate)
avg_rate_mbps=np.array(avg_rate_mbps); avg_snr_db=np.array(avg_snr_db)
bw_per_beam = W_total / beam_range / 1e6

opt_B   = beam_range[np.argmin(avg_aoi)]
opt_aoi = avg_aoi[opt_B - 1]
print(f"\nOptimal B★ = {opt_B},  Min Avg AoI = {opt_aoi:.3f} slots")

In [ ]:
# ============================================================
#  FIGURE 1 — Average AoI vs Number of Beams
# ============================================================
fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(beam_range, avg_aoi - std_aoi, avg_aoi + std_aoi,
                alpha=0.12, color='#3b82f6')
ax.plot(beam_range, avg_aoi, color='#3b82f6', linewidth=2.8,
        marker='o', markersize=6, zorder=4, label='Average AoI (±1σ band)')
ax.scatter([opt_B], [opt_aoi], color='#f59e0b', s=160, zorder=6,
           label=f'Optimal B★ = {opt_B}  (AoI = {opt_aoi:.2f} slots)')
ax.axvline(opt_B, color='#f59e0b', linewidth=1.2, linestyle='--', alpha=0.55)

mid_L = max(1, opt_B // 2)
mid_R = (C + opt_B) // 2
ax.annotate('Coverage bottleneck\n(cells unlit → AoI↑)',
            xy=(1.8, avg_aoi[1]), xytext=(mid_L, avg_aoi[1] + 0.8),
            color='#94a3b8', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#94a3b8', lw=1.2))
ax.annotate('Rate bottleneck\nW/B < R_th  → δ=0 → AoI↑',
            xy=(C - 0.5, avg_aoi[-1]), xytext=(mid_R - 1, avg_aoi[-1] + 0.5),
            color='#94a3b8', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#94a3b8', lw=1.2))
ax.text(opt_B + 0.15, opt_aoi + 0.15, f'B★={opt_B}', color='#f59e0b', fontsize=9)
ax.set_xlabel('Number of Active Beams  B', fontsize=12)
ax.set_ylabel('Average AoI  (time slots)', fontsize=12)
ax.set_title(
    f'Average AoI vs Number of Beams — LEO NB-IoT NTN\n'
    f'W={W_total/1e6:.0f}MHz  P={P_total_dBm}dBm({P_total_W:.0f}W)  '
    f'G_T={G_T_dBi}dBi  θ_3dB={theta_3dB}°  '
    f'G_rx={G_rx_dBi}dBi  R_th={R_th/1e6:.0f}Mbps  C={C}',
    fontsize=10, pad=12)
ax.set_xticks(beam_range)
ax.grid(True, alpha=0.5)
ax.legend(fontsize=9, labelcolor='#94a3b8')
plt.tight_layout()
plt.savefig('fig1_aoi_vs_beams.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("Saved: fig1_aoi_vs_beams.png")

In [ ]:
# ============================================================
#  FIGURE 2 — Supporting Metrics (2×2 panel)
# ============================================================
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.32)
axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]
for ax in axes:
    ax.grid(True, alpha=0.5); ax.set_xticks(beam_range)
    ax.axvline(opt_B, color='#f59e0b', linestyle='--', linewidth=1, alpha=0.55)

# A — Bandwidth per beam
axes[0].plot(beam_range, bw_per_beam, color='#a78bfa', linewidth=2.2, marker='D', markersize=5)
ref_snr = 10**(np.mean(avg_snr_db[avg_snr_db > 0])/10) if any(avg_snr_db > 0) else 10**(17/10)
thresh_bw = R_th / np.log2(1 + ref_snr) / 1e6
axes[0].axhline(thresh_bw, color='#f59e0b', linestyle=':', linewidth=1.5, alpha=0.8,
                label=f'Min BW for R≥R_th: {thresh_bw:.2f} MHz')
axes[0].set_ylabel('Bandwidth per Beam [MHz]')
axes[0].set_xlabel('Number of Beams B')
axes[0].set_title('Bandwidth per Beam  (W_total / B)  — 20 MHz NB-IoT')
axes[0].legend(fontsize=8, labelcolor='#94a3b8')

# B — Achievable rate
axes[1].plot(beam_range, avg_rate_mbps, color='#10b981', linewidth=2.2, marker='s', markersize=5)
axes[1].axhline(R_th/1e6, color='#ef4444', linestyle=':', linewidth=1.8,
                label=f'R_th={R_th/1e6:.0f} Mbps')
axes[1].fill_between(beam_range, avg_rate_mbps, R_th/1e6,
                     where=np.array(avg_rate_mbps) < R_th/1e6,
                     alpha=0.15, color='#ef4444', label='Rate < R_th (δ=0)')
axes[1].set_ylabel('Avg Rate [Mbps]  (served users)')
axes[1].set_xlabel('Number of Beams B')
axes[1].set_title('Avg Achievable Rate  (Eq.15: W/B·log₂(1+SNR))')
axes[1].legend(fontsize=8, labelcolor='#94a3b8')

# C — Success rate
bar_c = ['#10b981' if s>=succ_rate.max()*0.7 else
          '#f59e0b' if s>=succ_rate.max()*0.3 else '#ef4444' for s in succ_rate]
axes[2].bar(beam_range, succ_rate*100, color=bar_c, alpha=0.85, width=0.6)
axes[2].set_ylabel('Success Rate [%]  (δ=1 fraction)')
axes[2].set_xlabel('Number of Beams B')
axes[2].set_title('Successful Decoding Rate  (Eq.24)')

# D — Max AoI
axes[3].plot(beam_range, max_aoi_mean, color='#ef4444', linewidth=2.2,
             marker='^', markersize=5, label='Max AoI (mean over MC)')
axes[3].plot(beam_range, avg_aoi, color='#3b82f6', linewidth=2,
             marker='o', markersize=4, label='Avg AoI')
axes[3].set_ylabel('AoI (slots)')
axes[3].set_xlabel('Number of Beams B')
axes[3].set_title('Max vs Average AoI  (fairness indicator)')
axes[3].legend(fontsize=8, labelcolor='#94a3b8')

fig.suptitle('Supporting Metrics — LEO NB-IoT NTN AoI System  '
             f'(P=53dBm, G_T=16.2dBi, θ={theta_3dB}°, G_rx=16dBi, σ_sf=4dB+3dB margin)',
             fontsize=10, fontweight='bold', y=1.01)
plt.savefig('fig2_supporting_metrics.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("Saved: fig2_supporting_metrics.png")

In [ ]:
# ============================================================
#  FIGURE 3 — Sensitivity: Effect of R_th on B★
# ============================================================
R_th_sweep = [2e6, 5e6, 8e6, 10e6, 12e6]
pal        = ['#10b981','#06b6d4','#3b82f6','#f59e0b','#ef4444']
N_mc_s     = 300

fig, ax = plt.subplots(figsize=(11, 6))

for R_th_s, col in zip(R_th_sweep, pal):
    curve = []
    for B in beam_range:
        W_b = W_total/B; P_b = P_total_W/B
        sigma2_s = NSD_W_Hz * NF_lin * W_b
        mc = []
        for _ in range(N_mc_s):
            K_cells = [np.random.randint(K_min, K_max+1) for _ in range(C)]
            A = [[np.random.randint(1, aoi_init_max+1) for _ in range(K_cells[c])]
                 for c in range(C)]
            for t in range(T_slots):
                served = select_beams([cell_aoi_score(A[c]) for c in range(C)], B)
                for c in range(C):
                    K_c = K_cells[c]; P_user = P_b/K_c; new_A = []
                    for u in range(K_c):
                        delta = 0
                        if c in served:
                            xu,yu = user_position(cell_radius_km)
                            d = distance_3d(xu, yu, h_sat/1e3)
                            th_off = np.random.uniform(0, theta_3dB/2)
                            g = total_channel_gain(fc_GHz, d, sigma_sf_dB,
                                                   shadow_margin_dB, th_off,
                                                   G_T_dBi, theta_3dB)
                            snr = snr_mrt(P_user, g, sigma2_s)
                            R   = achievable_rate(W_b, snr)
                            delta = scheduling_indicator(R, R_th_s, True)
                        new_A.append(aoi_update(A[c][u], delta))
                    A[c] = new_A
            mc.append(np.mean([A[c][u] for c in range(C) for u in range(K_cells[c])]))
        curve.append(np.mean(mc))
    curve = np.array(curve)
    ob = beam_range[np.argmin(curve)]
    ax.plot(beam_range, curve, color=col, linewidth=2, marker='o', markersize=4,
            label=f'R_th={R_th_s/1e6:.0f} Mbps  (B★={ob})')
    ax.scatter([ob],[curve.min()], color=col, s=90, zorder=5)

ax.set_xlabel('Number of Active Beams  B', fontsize=12)
ax.set_ylabel('Average AoI  (time slots)', fontsize=12)
ax.set_title('R_th Sensitivity — Effect on Optimal B★\n'
             'Higher R_th → rate bottleneck earlier → B★ shifts left', fontsize=11)
ax.set_xticks(beam_range)
ax.grid(True, alpha=0.5)
ax.legend(fontsize=9, labelcolor='#94a3b8')
plt.tight_layout()
plt.savefig('fig3_rth_sensitivity.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("Saved: fig3_rth_sensitivity.png")

In [ ]:
print("=" * 65)
print("  RESULTS SUMMARY — Updated Parameters")
print("=" * 65)
print(f"  Paper parameters   : LEO_system_model__10_.pdf")
print(f"  Sources            : [A] 3GPP TR 36.763  [B] Kim et al.")
print(f"                       [C] IEEE AP Mag 2022  [D] Jiao TVT 2023")
print()
print(f"  Optimal B★         : {opt_B} beams")
print(f"  Min avg AoI        : {opt_aoi:.3f} slots")
print(f"  AoI at B=1         : {avg_aoi[0]:.3f}  (coverage bottleneck)")
print(f"  AoI at B=C={C}     : {avg_aoi[-1]:.3f}  (rate bottleneck)")
print(f"  BW/beam at B★      : {W_total/opt_B/1e6:.2f} MHz")
print(f"  Rate at B★         : {avg_rate_mbps[opt_B-1]:.2f} Mbps")
print(f"  Success rate @B★   : {succ_rate[opt_B-1]*100:.1f}%")
print()
print("  PARAMETER CHANGES SUMMARY:")
print(f"  G_T     : OLD 52.0 dBi   → NEW {G_T_dBi} dBi      [A]")
print(f"  θ_3dB   : OLD 0.4°       → NEW {theta_3dB}°    [A]")
print(f"  P_total : OLD 20 dBW     → NEW {P_total_dBW} dBW         [B]")
print(f"  W_total : OLD 500 MHz    → NEW {W_total/1e6:.0f} MHz          [A]")
print(f"  G_rx    : OLD Eq.7(A_rx) → NEW {G_rx_dBi} dBi direct  [C]")
print(f"  σ_sf    : OLD 2 dB       → NEW {sigma_sf_dB} dB              [B]")
print(f"  margin  : OLD 0 dB       → NEW +{shadow_margin_dB} dB fixed        [A]")
print(f"  Noise   : OLD k_B·T·NF·W → NEW NSD·NF·W (-174dBm/Hz)[B]")
print(f"  R_th    : OLD 700 Mbps   → NEW {R_th/1e6:.0f} Mbps          (recalib.)")
print()
print("  REMOVED: A_rx (aperture), old Eq.7 formula")
print("  NOT ADDED: Doppler shift (explicitly excluded per design)")